# Development corpus EDA

This notebook reads only the privacy-safe aggregate profile produced by `scripts/run_eda.py`. It does not open validation/test files or display raw comments.

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROFILE_PATH = PROJECT_ROOT / "data/reports/eda/development-profile.json"
if not PROFILE_PATH.is_file():
    message = "Run scripts/run_eda.py first"
    raise FileNotFoundError(message)
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))

## Corpus overview

In [ ]:
{key: profile[key] for key in [
    "records", "unique_texts", "duplicate_texts", "duplicate_groups",
    "unique_authors", "unique_videos", "unique_channels", "unique_queries",
]}

## Length, language, noise, and time aggregates

In [ ]:
{key: profile[key] for key in [
    "character_lengths", "token_lengths", "languages",
    "noise_categories", "monthly_records",
]}

## Interpretation checklist

- Review p50/p95/p99 lengths before selecting cleaning and embedding limits.
- Treat noise categories as overlapping diagnostics, not additive removal counts.
- Validate language labels manually on the local development-only sample before filtering.
- Keep validation and test unopened until their planned evaluation stages.
- Do not save cells containing raw comments into this tracked notebook.

## Stage 5 embedding artifacts

Load only the manifest and memory-mapped matrix here. Keep raw comment values out of the tracked notebook.

In [ ]:
import numpy as np

EMBEDDING_DIR = PROJECT_ROOT / "data/processed/embeddings"
embedding_manifest = json.loads((EMBEDDING_DIR / "embedding-manifest.json").read_text())
embeddings = np.load(EMBEDDING_DIR / "embeddings.npy", mmap_mode="r", allow_pickle=False)
{"shape": embeddings.shape, "dtype": str(embeddings.dtype), "model": embedding_manifest["model_name"]}

## Stage 7 final corpus

Stage 8 reads the deduplicated matrix through memory mapping and verifies its shape against the manifest.

In [ ]:
CORPUS_DIR = PROJECT_ROOT / "data/processed/corpus"
corpus_manifest = json.loads((CORPUS_DIR / "corpus-manifest.json").read_text())
final_embeddings = np.load(CORPUS_DIR / "final-embeddings.npy", mmap_mode="r", allow_pickle=False)
expected_shape = (corpus_manifest["stats"]["output_records"], corpus_manifest["dimensions"])
{
    "shape": final_embeddings.shape,
    "matches_manifest": final_embeddings.shape == expected_shape,
    "removed": corpus_manifest["stats"]["removed_semantic_duplicates"],
}

## Stage 8 UMAP visualization

The 2D space is diagnostic only. This plot uses coordinates and row indexes, never raw comment text.

In [ ]:
import matplotlib.pyplot as plt

UMAP_DIR = PROJECT_ROOT / "data/processed/umap"
visualization_manifest = json.loads((UMAP_DIR / "visualization-manifest.json").read_text())
coordinates = np.load(UMAP_DIR / "visualization-2d.npy", mmap_mode="r", allow_pickle=False)
plot_limit = min(len(coordinates), 50_000)
figure, axis = plt.subplots(figsize=(10, 7))
axis.scatter(coordinates[:plot_limit, 0], coordinates[:plot_limit, 1], s=2, alpha=0.25)
axis.set(title="UMAP diagnostic projection", xlabel="UMAP-1", ylabel="UMAP-2")
figure